In [ ]:
"""
==================================================
ML LEARNING JOURNEY - DAY 93
==================================================
Week: 14 of 24
Day: 93 of 168
Date: May 10, 2026 
Topic: Dockerize the Security System

Week 14 Progress:
✅ Day 92: Docker Fundamentals
🔄 Day 93: Dockerize Security System (TODAY!)
⬜ Day 94: RESTful API Design & FastAPI Deep Dive
⬜ Day 95: API Documentation & Authentication
⬜ Day 96: CI/CD & GitHub Actions
⬜ Day 97: Automated Testing & Deployment
⬜ Day 98: Model Monitoring & Logging

Progress: 14.3% (1/7 days)

==================================================
🎯 Week 14 Project: MLOps & Production Engineering
- Containerize ML projects with Docker
- Build production-grade APIs with FastAPI
- Automate deployments with GitHub Actions CI/CD
- Add monitoring, logging, and error tracking

🎯 Today's Learning Objectives:
1. Write Dockerfile.api for the FastAPI detection backend
2. Write Dockerfile.streamlit for the Streamlit UI
3. Implement the full docker-compose.yml designed on Day 92
4. Test the containerized system end-to-end
5. Optimize image sizes and container startup times

📚 Today's Structure:
   Part 1 (1.5h): Audit the Security System codebase for Docker readiness
   Part 2 (2h):   Write Dockerfiles for API and Streamlit services
   Part 3 (1.5h): Implement docker-compose.yml and wire services together
   Part 4 (1h):   End-to-end testing and troubleshooting

🎯 SUCCESS CRITERIA:
   ✅ Dockerfile.api builds successfully for FastAPI backend
   ✅ Dockerfile.streamlit builds successfully for Streamlit UI
   ✅ docker compose up starts all 3 services without errors
   ✅ Streamlit UI can communicate with FastAPI backend inside Docker network
   ✅ Model weights loaded from mounted volume (not baked into image)
   ✅ Detection results and logs persist to host machine via volumes

==================================================
"""

In [1]:
# ==================================================
# INSTALL REQUIRED LIBRARIES
# ==================================================

import sys
!{sys.executable} -m pip install docker requests -q

print("✅ Libraries installed!")

print("\n" + "=" * 80)

# ==================================================
# IMPORT LIBRARIES
# ==================================================

print("\n" + "=" * 80)
print("📚 IMPORTING LIBRARIES")
print("=" * 80)

import os
import subprocess
import json
import time
import requests

print("\n✅ All libraries imported successfully!")
print("=" * 80)

'c:\Program' is not recognized as an internal or external command,
operable program or batch file.


✅ Libraries installed!


📚 IMPORTING LIBRARIES

✅ All libraries imported successfully!


In [2]:
print("\n" + "=" * 80)
print("🔍 PART 1: SECURITY SYSTEM CODEBASE AUDIT")
print("=" * 80)


🔍 PART 1: SECURITY SYSTEM CODEBASE AUDIT


In [3]:
# ==================================================
# EXERCISE 1.1: REVIEW THE SECURITY SYSTEM STRUCTURE
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.1: Security System Structure Review")
print("=" * 80)

"""
📖 THEORY: Pre-Docker Checklist

Before writing a Dockerfile, audit the project for:
1. Entry points — what command starts the app?
2. Dependencies — what's in requirements.txt?
3. Hard-coded paths — any absolute paths that break inside a container?
4. Large files — model weights, videos, datasets (these become volumes)
5. Environment variables — secrets or config that shouldn't be hardcoded
6. Ports — what port does the app listen on?
7. External services — does it connect to a database, Redis, another API?
"""

print("""
📂 Expected Security System structure (week4_detection_tracking):

  week4_detection_tracking/
  ├── app.py                    ← Streamlit frontend entry point
  ├── api/
  │   ├── main.py               ← FastAPI backend entry point
  │   ├── routes/
  │   │   ├── detection.py      ← Detection endpoints
  │   │   └── tracking.py       ← Tracking endpoints
  │   └── models/
  │       └── detector.py       ← YOLO model wrapper
  ├── models/
  │   └── best.pt               ← YOLOv8 weights (VOLUME — do not bake in)
  ├── videos/                   ← Input videos (VOLUME)
  ├── results/                  ← Output frames/videos (VOLUME)
  ├── logs/                     ← App logs (VOLUME)
  ├── requirements.txt          ← Python dependencies
  └── config.py                 ← Configuration / env var loading
""")

print("""
✅ PRE-DOCKER CHECKLIST FOR SECURITY SYSTEM:

  Entry points:
  □ Streamlit UI    → streamlit run app.py --server.address=0.0.0.0
  □ FastAPI backend → uvicorn api.main:app --host 0.0.0.0 --port 8000

  Large files to mount as volumes (NOT baked into image):
  □ models/best.pt       → /app/models  (YOLOv8 weights, ~6MB–25MB)
  □ videos/              → /app/videos  (input video files)
  □ results/             → /app/results (detection output)
  □ logs/                → /app/logs    (runtime logs)

  Environment variables to externalize:
  □ MODEL_PATH           → path to YOLO weights inside container
  □ API_URL              → FastAPI URL (used by Streamlit to call the API)
  □ CONFIDENCE_THRESHOLD → detection confidence (default 0.5)
  □ LOG_LEVEL            → INFO / DEBUG / WARNING

  Ports:
  □ FastAPI  → 8000
  □ Streamlit → 8501

  Hard-coded paths to fix:
  □ Any os.path referencing local absolute paths
  □ Replace with os.environ.get("MODEL_PATH", "models/best.pt")
""")

print("\n✅ Exercise 1.1 Complete!")
print("=" * 80)


EXERCISE 1.1: Security System Structure Review

📂 Expected Security System structure (week4_detection_tracking):

  week4_detection_tracking/
  ├── app.py                    ← Streamlit frontend entry point
  ├── api/
  │   ├── main.py               ← FastAPI backend entry point
  │   ├── routes/
  │   │   ├── detection.py      ← Detection endpoints
  │   │   └── tracking.py       ← Tracking endpoints
  │   └── models/
  │       └── detector.py       ← YOLO model wrapper
  ├── models/
  │   └── best.pt               ← YOLOv8 weights (VOLUME — do not bake in)
  ├── videos/                   ← Input videos (VOLUME)
  ├── results/                  ← Output frames/videos (VOLUME)
  ├── logs/                     ← App logs (VOLUME)
  ├── requirements.txt          ← Python dependencies
  └── config.py                 ← Configuration / env var loading


✅ PRE-DOCKER CHECKLIST FOR SECURITY SYSTEM:

  Entry points:
  □ Streamlit UI    → streamlit run app.py --server.address=0.0.0.0
  □ FastAPI

In [4]:
# ==================================================
# EXERCISE 1.2: REQUIREMENTS.TXT FOR DOCKER
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.2: Requirements.txt Audit for Docker")
print("=" * 80)

"""
📖 THEORY: Pinning Dependencies

In Docker, always pin exact versions in requirements.txt.
Without pinning, a rebuild 6 months later may pull a different version
and break your app silently.

Use pip freeze > requirements.txt in your working venv to get exact versions.
"""

requirements_api = '''
# requirements.api.txt — FastAPI Backend
# Generated from working virtual environment

# Core API framework
fastapi==0.111.0
uvicorn[standard]==0.29.0
python-multipart==0.0.9

# ML / Computer Vision
torch==2.2.2
torchvision==0.17.2
ultralytics==8.2.0          # YOLOv8
opencv-python-headless==4.9.0.80   # headless = no GUI (important for Docker!)
numpy==1.26.4
Pillow==10.3.0

# Tracking
deep-sort-realtime==1.3.2

# Caching
redis==5.0.4

# Utilities
python-dotenv==1.0.1
pydantic==2.7.1
loguru==0.7.2
'''

requirements_streamlit = '''
# requirements.streamlit.txt — Streamlit UI
# Generated from working virtual environment

# Frontend
streamlit==1.33.0

# API communication
requests==2.31.0
httpx==0.27.0

# Visualization
opencv-python-headless==4.9.0.80
numpy==1.26.4
Pillow==10.3.0
plotly==5.21.0

# Utilities
python-dotenv==1.0.1
'''

print("📄 requirements.api.txt:")
print(requirements_api)

print("📄 requirements.streamlit.txt:")
print(requirements_streamlit)

print("""
💡 NOTE: opencv-python-headless vs opencv-python

  opencv-python        → includes GUI support (needs display server)
  opencv-python-headless → no GUI, lighter weight

  Always use headless in Docker — containers have no display server.
  Using the wrong one causes: "cannot connect to X server" errors.
""")

print("\n✅ Exercise 1.2 Complete!")
print("=" * 80)


EXERCISE 1.2: Requirements.txt Audit for Docker
📄 requirements.api.txt:

# requirements.api.txt — FastAPI Backend
# Generated from working virtual environment

# Core API framework
fastapi==0.111.0
uvicorn[standard]==0.29.0
python-multipart==0.0.9

# ML / Computer Vision
torch==2.2.2
torchvision==0.17.2
ultralytics==8.2.0          # YOLOv8
opencv-python-headless==4.9.0.80   # headless = no GUI (important for Docker!)
numpy==1.26.4
Pillow==10.3.0

# Tracking
deep-sort-realtime==1.3.2

# Caching
redis==5.0.4

# Utilities
python-dotenv==1.0.1
pydantic==2.7.1
loguru==0.7.2

📄 requirements.streamlit.txt:

# requirements.streamlit.txt — Streamlit UI
# Generated from working virtual environment

# Frontend
streamlit==1.33.0

# API communication
requests==2.31.0
httpx==0.27.0

# Visualization
opencv-python-headless==4.9.0.80
numpy==1.26.4
Pillow==10.3.0
plotly==5.21.0

# Utilities
python-dotenv==1.0.1


💡 NOTE: opencv-python-headless vs opencv-python

  opencv-python        → includes GUI sup

In [5]:
print("\n" + "=" * 80)
print("🐋 PART 2: WRITING THE DOCKERFILES")
print("=" * 80)


🐋 PART 2: WRITING THE DOCKERFILES


In [6]:
# ==================================================
# EXERCISE 2.1: DOCKERFILE FOR FASTAPI BACKEND
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 2.1: Dockerfile.api — FastAPI Detection Backend")
print("=" * 80)

dockerfile_api = '''
# ============================================================
# Dockerfile.api — FastAPI Detection & Tracking Backend
# AI Security System — Week 14, Day 93
# ============================================================

# ── STAGE 1: Builder ─────────────────────────────────────────
FROM python:3.10-slim AS builder

WORKDIR /build

# Install build tools
RUN apt-get update && apt-get install -y --no-install-recommends \\
    gcc \\
    g++ \\
    && rm -rf /var/lib/apt/lists/*

# Copy and install Python dependencies
COPY requirements.api.txt .
RUN pip install --no-cache-dir --prefix=/install -r requirements.api.txt


# ── STAGE 2: Runtime ─────────────────────────────────────────
FROM python:3.10-slim AS runtime

# Runtime system dependencies for OpenCV and PyTorch
RUN apt-get update && apt-get install -y --no-install-recommends \\
    libgl1-mesa-glx \\
    libglib2.0-0 \\
    libsm6 \\
    libxext6 \\
    libxrender-dev \\
    libgomp1 \\
    curl \\
    && rm -rf /var/lib/apt/lists/*

WORKDIR /app

# Copy installed packages from builder
COPY --from=builder /install /usr/local

# Copy application source code
COPY api/ ./api/
COPY config.py .

# Create directories for mounted volumes
RUN mkdir -p models videos results logs

# Environment variables (defaults — override in docker-compose.yml)
ENV MODEL_PATH=/app/models/best.pt
ENV CONFIDENCE_THRESHOLD=0.5
ENV LOG_LEVEL=INFO
ENV PYTHONUNBUFFERED=1
ENV PYTHONDONTWRITEBYTECODE=1

# Health check — polls the /health endpoint every 30s
HEALTHCHECK --interval=30s --timeout=10s --start-period=60s --retries=3 \\
    CMD curl -f http://localhost:8000/health || exit 1

EXPOSE 8000

CMD ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "1"]
'''

print("📄 Dockerfile.api:")
print(dockerfile_api)

print("""
💡 Notes on this Dockerfile:

  --start-period=60s in HEALTHCHECK:
    YOLOv8 model loading takes 10-30 seconds on first start.
    Without start_period, Docker marks the container unhealthy before
    the model even finishes loading and kills it unnecessarily.

  --workers 1:
    Keep to 1 worker when loading large ML models.
    Multiple workers = multiple model copies in memory = OOM errors.

  libgomp1:
    Required by PyTorch for OpenMP parallel processing.
    Missing it causes: "libgomp.so.1: cannot open shared object file"
""")

print("\n✅ Exercise 2.1 Complete!")
print("=" * 80)


EXERCISE 2.1: Dockerfile.api — FastAPI Detection Backend
📄 Dockerfile.api:

# ============================================================
# Dockerfile.api — FastAPI Detection & Tracking Backend
# AI Security System — Week 14, Day 93
# ============================================================

# ── STAGE 1: Builder ─────────────────────────────────────────
FROM python:3.10-slim AS builder

WORKDIR /build

# Install build tools
RUN apt-get update && apt-get install -y --no-install-recommends \
    gcc \
    g++ \
    && rm -rf /var/lib/apt/lists/*

# Copy and install Python dependencies
COPY requirements.api.txt .
RUN pip install --no-cache-dir --prefix=/install -r requirements.api.txt


# ── STAGE 2: Runtime ─────────────────────────────────────────
FROM python:3.10-slim AS runtime

# Runtime system dependencies for OpenCV and PyTorch
RUN apt-get update && apt-get install -y --no-install-recommends \
    libgl1-mesa-glx \
    libglib2.0-0 \
    libsm6 \
    libxext6 \
    libxrende

In [7]:
# ==================================================
# EXERCISE 2.2: DOCKERFILE FOR STREAMLIT UI
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 2.2: Dockerfile.streamlit — Streamlit Frontend")
print("=" * 80)

dockerfile_streamlit = '''
# ============================================================
# Dockerfile.streamlit — Streamlit UI Frontend
# AI Security System — Week 14, Day 93
# ============================================================

FROM python:3.10-slim

# System dependencies
RUN apt-get update && apt-get install -y --no-install-recommends \\
    libgl1-mesa-glx \\
    libglib2.0-0 \\
    curl \\
    && rm -rf /var/lib/apt/lists/*

WORKDIR /app

# Install Python dependencies
COPY requirements.streamlit.txt .
RUN pip install --no-cache-dir -r requirements.streamlit.txt

# Copy Streamlit app code
COPY app.py .
COPY config.py .

# Create results directory (read from API output)
RUN mkdir -p results

# Streamlit configuration
ENV STREAMLIT_SERVER_PORT=8501
ENV STREAMLIT_SERVER_HEADLESS=true
ENV STREAMLIT_SERVER_ENABLE_CORS=false
ENV STREAMLIT_SERVER_ADDRESS=0.0.0.0

# API connection (overridden in docker-compose.yml)
ENV API_URL=http://fastapi_backend:8000

ENV PYTHONUNBUFFERED=1

HEALTHCHECK --interval=30s --timeout=10s --start-period=30s --retries=3 \\
    CMD curl -f http://localhost:8501/_stcore/health || exit 1

EXPOSE 8501

CMD ["streamlit", "run", "app.py"]
'''

print("📄 Dockerfile.streamlit:")
print(dockerfile_streamlit)

print("""
💡 Notes on this Dockerfile:

  ENV API_URL=http://fastapi_backend:8000:
    Inside Docker network, containers communicate by SERVICE NAME.
    "fastapi_backend" resolves to the FastAPI container's IP automatically.
    This is why we don't use localhost or 127.0.0.1 between containers.

  Streamlit health check endpoint:
    /_stcore/health is Streamlit's built-in health check route.
    Available in Streamlit >= 1.18.0.

  No multi-stage build here:
    Streamlit has no heavy compile-time dependencies.
    Single-stage is sufficient and simpler.
""")

print("\n✅ Exercise 2.2 Complete!")
print("=" * 80)


EXERCISE 2.2: Dockerfile.streamlit — Streamlit Frontend
📄 Dockerfile.streamlit:

# ============================================================
# Dockerfile.streamlit — Streamlit UI Frontend
# AI Security System — Week 14, Day 93
# ============================================================

FROM python:3.10-slim

# System dependencies
RUN apt-get update && apt-get install -y --no-install-recommends \
    libgl1-mesa-glx \
    libglib2.0-0 \
    curl \
    && rm -rf /var/lib/apt/lists/*

WORKDIR /app

# Install Python dependencies
COPY requirements.streamlit.txt .
RUN pip install --no-cache-dir -r requirements.streamlit.txt

# Copy Streamlit app code
COPY app.py .
COPY config.py .

# Create results directory (read from API output)
RUN mkdir -p results

# Streamlit configuration
ENV STREAMLIT_SERVER_PORT=8501
ENV STREAMLIT_SERVER_HEADLESS=true
ENV STREAMLIT_SERVER_ENABLE_CORS=false
ENV STREAMLIT_SERVER_ADDRESS=0.0.0.0

# API connection (overridden in docker-compose.yml)
ENV API_URL=ht

In [8]:
# ==================================================
# EXERCISE 2.3: .DOCKERIGNORE FILE
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 2.3: .dockerignore for Security System")
print("=" * 80)

dockerignore = '''
# ============================================================
# .dockerignore — AI Security System
# ============================================================

# Python cache
__pycache__/
*.pyc
*.pyo
*.pyd
.Python

# Virtual environments
venv/
env/
.venv/
ENV/

# Version control
.git/
.gitignore
.gitattributes

# Large model weights — mounted as volume at runtime
models/
*.pt
*.pth
*.onnx
*.h5

# Video files — mounted as volume at runtime
videos/
*.mp4
*.avi
*.mov
*.mkv

# Output files — mounted as volume at runtime
results/

# Logs — mounted as volume at runtime
logs/
*.log

# Jupyter notebooks
*.ipynb
.ipynb_checkpoints/

# Tests and docs
tests/
docs/
*.md

# Environment files (secrets!)
.env
.env.*

# OS files
.DS_Store
Thumbs.db

# IDE
.vscode/
.idea/
'''

print("📄 .dockerignore:")
print(dockerignore)

print("""
⏱️  Build context size comparison (estimate):

  Without .dockerignore:   ~2.1 GB  (includes model weights + videos)
  With .dockerignore:      ~45 MB   (source code + requirements only)

  Result: builds are ~46x faster just from ignoring the right files.
""")

print("\n✅ Exercise 2.3 Complete!")
print("=" * 80)


EXERCISE 2.3: .dockerignore for Security System
📄 .dockerignore:

# ============================================================
# .dockerignore — AI Security System
# ============================================================

# Python cache
__pycache__/
*.pyc
*.pyo
*.pyd
.Python

# Virtual environments
venv/
env/
.venv/
ENV/

# Version control
.git/
.gitignore
.gitattributes

# Large model weights — mounted as volume at runtime
models/
*.pt
*.pth
*.onnx
*.h5

# Video files — mounted as volume at runtime
videos/
*.mp4
*.avi
*.mov
*.mkv

# Output files — mounted as volume at runtime
results/

# Logs — mounted as volume at runtime
logs/
*.log

# Jupyter notebooks
*.ipynb
.ipynb_checkpoints/

# Tests and docs
tests/
docs/
*.md

# Environment files (secrets!)
.env
.env.*

# OS files
.DS_Store
Thumbs.db

# IDE
.vscode/
.idea/


⏱️  Build context size comparison (estimate):

  Without .dockerignore:   ~2.1 GB  (includes model weights + videos)
  With .dockerignore:      ~45 MB   (source 

In [9]:
print("\n" + "=" * 80)
print("🔧 PART 3: DOCKER COMPOSE & WIRING SERVICES TOGETHER")
print("=" * 80)


🔧 PART 3: DOCKER COMPOSE & WIRING SERVICES TOGETHER


In [10]:
# ==================================================
# EXERCISE 3.1: FULL DOCKER-COMPOSE.YML
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 3.1: Full docker-compose.yml Implementation")
print("=" * 80)

docker_compose = '''
# ============================================================
# docker-compose.yml — AI Security System
# Week 14, Day 93
# ============================================================

version: "3.9"

services:

  # ── FastAPI Detection & Tracking Backend ─────────────────
  fastapi_backend:
    build:
      context: .
      dockerfile: Dockerfile.api
    container_name: security_api
    ports:
      - "8000:8000"
    volumes:
      - ./models:/app/models:ro       # Model weights (read-only)
      - ./videos:/app/videos:ro       # Input videos (read-only)
      - ./results:/app/results        # Output (read-write)
      - ./logs:/app/logs              # Logs (read-write)
    environment:
      - MODEL_PATH=/app/models/best.pt
      - CONFIDENCE_THRESHOLD=0.5
      - REDIS_URL=redis://redis_cache:6379/0
      - LOG_LEVEL=INFO
    env_file:
      - .env                          # Secrets override (git-ignored)
    depends_on:
      redis_cache:
        condition: service_healthy
    restart: unless-stopped
    deploy:
      resources:
        limits:
          memory: 4G
    networks:
      - security_network

  # ── Streamlit UI ──────────────────────────────────────────
  streamlit_ui:
    build:
      context: .
      dockerfile: Dockerfile.streamlit
    container_name: security_ui
    ports:
      - "8501:8501"
    volumes:
      - ./results:/app/results:ro     # Read detection results
    environment:
      - API_URL=http://fastapi_backend:8000
      - LOG_LEVEL=INFO
    depends_on:
      fastapi_backend:
        condition: service_healthy
    restart: unless-stopped
    networks:
      - security_network

  # ── Redis Cache ───────────────────────────────────────────
  redis_cache:
    image: redis:7-alpine
    container_name: security_redis
    ports:
      - "6379:6379"
    volumes:
      - redis_data:/data
    command: redis-server --maxmemory 256mb --maxmemory-policy allkeys-lru
    healthcheck:
      test: ["CMD", "redis-cli", "ping"]
      interval: 10s
      timeout: 5s
      retries: 5
    restart: unless-stopped
    networks:
      - security_network

# ── Named Volumes ─────────────────────────────────────────
volumes:
  redis_data:
    driver: local

# ── Networks ──────────────────────────────────────────────
networks:
  security_network:
    driver: bridge
    name: security_network
'''

print("📄 docker-compose.yml:")
print(docker_compose)

print("\n✅ Exercise 3.1 Complete!")
print("=" * 80)


EXERCISE 3.1: Full docker-compose.yml Implementation
📄 docker-compose.yml:

# ============================================================
# docker-compose.yml — AI Security System
# Week 14, Day 93
# ============================================================

version: "3.9"

services:

  # ── FastAPI Detection & Tracking Backend ─────────────────
  fastapi_backend:
    build:
      context: .
      dockerfile: Dockerfile.api
    container_name: security_api
    ports:
      - "8000:8000"
    volumes:
      - ./models:/app/models:ro       # Model weights (read-only)
      - ./videos:/app/videos:ro       # Input videos (read-only)
      - ./results:/app/results        # Output (read-write)
      - ./logs:/app/logs              # Logs (read-write)
    environment:
      - MODEL_PATH=/app/models/best.pt
      - CONFIDENCE_THRESHOLD=0.5
      - REDIS_URL=redis://redis_cache:6379/0
      - LOG_LEVEL=INFO
    env_file:
      - .env                          # Secrets override (git-ignored)

In [11]:
# ==================================================
# EXERCISE 3.2: CONFIG.PY UPDATES FOR DOCKER
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 3.2: config.py Updates for Docker Compatibility")
print("=" * 80)

"""
📖 THEORY: 12-Factor App Config

The 12-Factor App methodology says: store config in environment variables.
This makes your app work identically in dev (local), staging, and production
without changing any code — just change the environment variables.
"""

config_py = '''
# ============================================================
# config.py — Environment-aware configuration
# Works locally AND inside Docker containers
# ============================================================

import os
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────
# Use env var if set (Docker), otherwise fall back to local path
BASE_DIR = Path(__file__).parent

MODEL_PATH = os.environ.get("MODEL_PATH", str(BASE_DIR / "models" / "best.pt"))
VIDEOS_DIR = os.environ.get("VIDEOS_DIR", str(BASE_DIR / "videos"))
RESULTS_DIR = os.environ.get("RESULTS_DIR", str(BASE_DIR / "results"))
LOGS_DIR = os.environ.get("LOGS_DIR", str(BASE_DIR / "logs"))

# ── Detection Settings ────────────────────────────────────
CONFIDENCE_THRESHOLD = float(os.environ.get("CONFIDENCE_THRESHOLD", "0.5"))
IOU_THRESHOLD = float(os.environ.get("IOU_THRESHOLD", "0.45"))
MAX_DETECTIONS = int(os.environ.get("MAX_DETECTIONS", "100"))

# ── API Settings ──────────────────────────────────────────
API_HOST = os.environ.get("API_HOST", "0.0.0.0")
API_PORT = int(os.environ.get("API_PORT", "8000"))
API_URL = os.environ.get("API_URL", "http://localhost:8000")

# ── Redis ─────────────────────────────────────────────────
REDIS_URL = os.environ.get("REDIS_URL", "redis://localhost:6379/0")

# ── Logging ───────────────────────────────────────────────
LOG_LEVEL = os.environ.get("LOG_LEVEL", "INFO")

# ── Ensure output dirs exist ──────────────────────────────
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)
'''

print("📄 config.py:")
print(config_py)

env_example = '''
# ============================================================
# .env.example — copy to .env and fill in values
# (add .env to .gitignore — never commit secrets!)
# ============================================================

MODEL_PATH=/app/models/best.pt
CONFIDENCE_THRESHOLD=0.5
IOU_THRESHOLD=0.45
LOG_LEVEL=INFO
REDIS_URL=redis://redis_cache:6379/0
'''

print("📄 .env.example:")
print(env_example)

print("\n✅ Exercise 3.2 Complete!")
print("=" * 80)


EXERCISE 3.2: config.py Updates for Docker Compatibility
📄 config.py:

# ============================================================
# config.py — Environment-aware configuration
# Works locally AND inside Docker containers
# ============================================================

import os
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────
# Use env var if set (Docker), otherwise fall back to local path
BASE_DIR = Path(__file__).parent

MODEL_PATH = os.environ.get("MODEL_PATH", str(BASE_DIR / "models" / "best.pt"))
VIDEOS_DIR = os.environ.get("VIDEOS_DIR", str(BASE_DIR / "videos"))
RESULTS_DIR = os.environ.get("RESULTS_DIR", str(BASE_DIR / "results"))
LOGS_DIR = os.environ.get("LOGS_DIR", str(BASE_DIR / "logs"))

# ── Detection Settings ────────────────────────────────────
CONFIDENCE_THRESHOLD = float(os.environ.get("CONFIDENCE_THRESHOLD", "0.5"))
IOU_THRESHOLD = float(os.environ.get("IOU_THRESHOLD", "0.45"))
MAX_DETECTIONS = int(os.environ.

In [12]:
print("\n" + "=" * 80)
print("🧪 PART 4: TESTING THE CONTAINERIZED SYSTEM")
print("=" * 80)


🧪 PART 4: TESTING THE CONTAINERIZED SYSTEM


In [13]:
# ==================================================
# EXERCISE 4.1: BUILD AND RUN COMMANDS
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 4.1: Build & Run the Containers")
print("=" * 80)

print("""
📋 STEP-BY-STEP: Running the Security System in Docker

Open a terminal in your week4_detection_tracking folder and run these in order:

  ─── Step 1: Verify Docker is running ───────────────────────
  docker --version
  docker compose version

  ─── Step 2: Build all images ────────────────────────────────
  docker compose build

  # Or rebuild a single service:
  docker compose build fastapi_backend

  ─── Step 3: Start all services ──────────────────────────────
  docker compose up

  # Or start in background (detached mode):
  docker compose up -d

  ─── Step 4: Verify all services are running ────────────────
  docker compose ps

  Expected output:
  NAME               STATUS          PORTS
  security_redis     Up (healthy)    0.0.0.0:6379->6379/tcp
  security_api       Up (healthy)    0.0.0.0:8000->8000/tcp
  security_ui        Up (healthy)    0.0.0.0:8501->8501/tcp

  ─── Step 5: Test the endpoints ─────────────────────────────
  # FastAPI health check
  curl http://localhost:8000/health

  # FastAPI docs (interactive Swagger UI)
  Open browser → http://localhost:8000/docs

  # Streamlit UI
  Open browser → http://localhost:8501

  ─── Step 6: View logs ───────────────────────────────────────
  docker compose logs                    # All services
  docker compose logs fastapi_backend    # Single service
  docker compose logs -f fastapi_backend # Follow (live tail)

  ─── Step 7: Stop everything ─────────────────────────────────
  docker compose down          # Stop and remove containers
  docker compose down -v       # Also remove named volumes
""")

print("\n✅ Exercise 4.1 Complete!")
print("=" * 80)


EXERCISE 4.1: Build & Run the Containers

📋 STEP-BY-STEP: Running the Security System in Docker

Open a terminal in your week4_detection_tracking folder and run these in order:

  ─── Step 1: Verify Docker is running ───────────────────────
  docker --version
  docker compose version

  ─── Step 2: Build all images ────────────────────────────────
  docker compose build

  # Or rebuild a single service:
  docker compose build fastapi_backend

  ─── Step 3: Start all services ──────────────────────────────
  docker compose up

  # Or start in background (detached mode):
  docker compose up -d

  ─── Step 4: Verify all services are running ────────────────
  docker compose ps

  Expected output:
  NAME               STATUS          PORTS
  security_redis     Up (healthy)    0.0.0.0:6379->6379/tcp
  security_api       Up (healthy)    0.0.0.0:8000->8000/tcp
  security_ui        Up (healthy)    0.0.0.0:8501->8501/tcp

  ─── Step 5: Test the endpoints ─────────────────────────────
  # FastA

In [14]:
# ==================================================
# EXERCISE 4.2: COMMON ERRORS & FIXES
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 4.2: Common Docker Errors & How to Fix Them")
print("=" * 80)

print("""
🔧 TROUBLESHOOTING GUIDE:

  ─── Error 1: Port already in use ────────────────────────────
  Error: "Bind for 0.0.0.0:8000 failed: port is already allocated"
  Fix:
    # Find what's using the port
    netstat -ano | findstr :8000          # Windows
    lsof -i :8000                         # Mac/Linux

    # Or change the host port in docker-compose.yml:
    ports:
      - "8001:8000"                       # Use 8001 on host instead


  ─── Error 2: Model not found ─────────────────────────────────
  Error: "FileNotFoundError: models/best.pt not found"
  Fix:
    # Ensure models/ folder exists locally with the weights file:
    ls week4_detection_tracking/models/

    # Check the volume mount in docker-compose.yml:
    volumes:
      - ./models:/app/models:ro


  ─── Error 3: Cannot connect to X server (OpenCV) ─────────────
  Error: "cannot connect to X server"
  Fix:
    # Wrong OpenCV package — switch to headless:
    # In requirements.api.txt, replace:
    opencv-python → opencv-python-headless


  ─── Error 4: Streamlit can't reach FastAPI ───────────────────
  Error: "Connection refused: http://localhost:8000"
  Fix:
    # In Docker, services talk by SERVICE NAME, not localhost
    # In docker-compose.yml:
    environment:
      - API_URL=http://fastapi_backend:8000   # ✅ correct
      # - API_URL=http://localhost:8000       # ❌ wrong inside Docker


  ─── Error 5: Container exits immediately ─────────────────────
  Error: Container shows "Exited (1)"
  Fix:
    docker compose logs fastapi_backend    # Read the actual error
    # Common causes: missing env var, import error, missing file


  ─── Error 6: Out of memory ───────────────────────────────────
  Error: Container killed, "OOMKilled" in docker inspect
  Fix:
    # Increase memory limit in docker-compose.yml:
    deploy:
      resources:
        limits:
          memory: 6G                        # Increase from 4G

    # Or reduce: use --workers 1 (not multiple Uvicorn workers)
""")

print("\n✅ Exercise 4.2 Complete!")
print("=" * 80)


EXERCISE 4.2: Common Docker Errors & How to Fix Them

🔧 TROUBLESHOOTING GUIDE:

  ─── Error 1: Port already in use ────────────────────────────
  Error: "Bind for 0.0.0.0:8000 failed: port is already allocated"
  Fix:
    # Find what's using the port
    netstat -ano | findstr :8000          # Windows
    lsof -i :8000                         # Mac/Linux

    # Or change the host port in docker-compose.yml:
    ports:
      - "8001:8000"                       # Use 8001 on host instead


  ─── Error 2: Model not found ─────────────────────────────────
  Error: "FileNotFoundError: models/best.pt not found"
  Fix:
    # Ensure models/ folder exists locally with the weights file:
    ls week4_detection_tracking/models/

    # Check the volume mount in docker-compose.yml:
    volumes:
      - ./models:/app/models:ro


  ─── Error 3: Cannot connect to X server (OpenCV) ─────────────
  Error: "cannot connect to X server"
  Fix:
    # Wrong OpenCV package — switch to headless:
    # In requi

In [15]:
# ==================================================
# EXERCISE 4.3: DAY 93 SUMMARY
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 4.3: Day 93 Summary")
print("=" * 80)

print("""
📚 WHAT WE BUILT TODAY:

✅ Pre-Docker Audit:
   • Identified all entry points (FastAPI + Streamlit)
   • Separated requirements into api and streamlit sets
   • Identified large files to mount as volumes (models, videos, results, logs)
   • Switched opencv-python → opencv-python-headless for Docker

✅ Dockerfile.api (FastAPI Backend):
   • Multi-stage build: builder + runtime stages
   • All system dependencies for OpenCV + PyTorch (libgomp1!)
   • Health check with 60s start_period for model loading time
   • Single Uvicorn worker to avoid OOM with large ML models

✅ Dockerfile.streamlit (Streamlit UI):
   • Single-stage (no heavy compile deps)
   • Service name API_URL: http://fastapi_backend:8000
   • Built-in Streamlit health check endpoint

✅ .dockerignore:
   • Excluded model weights, videos, results, logs, .git
   • Reduced build context from ~2.1GB → ~45MB

✅ docker-compose.yml:
   • 3 services: fastapi_backend, streamlit_ui, redis_cache
   • Health check conditions for startup ordering
   • Named volumes for Redis persistence
   • Custom bridge network for inter-service communication

✅ config.py:
   • All paths and settings read from environment variables
   • Fallback defaults for local development
   • Works identically inside and outside Docker

💡 KEY LESSONS:
   1. opencv-python-headless is mandatory in Docker (no display server)
   2. Services talk by SERVICE NAME inside Docker network, not localhost
   3. Health checks with start_period prevent premature container restarts
   4. Mount model weights as :ro volumes — never bake them into the image
   5. .dockerignore cuts build context size by 97% for ML projects
""")

print("\n✅ Exercise 4.3 Complete!")
print("=" * 80)


EXERCISE 4.3: Day 93 Summary

📚 WHAT WE BUILT TODAY:

✅ Pre-Docker Audit:
   • Identified all entry points (FastAPI + Streamlit)
   • Separated requirements into api and streamlit sets
   • Identified large files to mount as volumes (models, videos, results, logs)
   • Switched opencv-python → opencv-python-headless for Docker

✅ Dockerfile.api (FastAPI Backend):
   • Multi-stage build: builder + runtime stages
   • All system dependencies for OpenCV + PyTorch (libgomp1!)
   • Health check with 60s start_period for model loading time
   • Single Uvicorn worker to avoid OOM with large ML models

✅ Dockerfile.streamlit (Streamlit UI):
   • Single-stage (no heavy compile deps)
   • Service name API_URL: http://fastapi_backend:8000
   • Built-in Streamlit health check endpoint

✅ .dockerignore:
   • Excluded model weights, videos, results, logs, .git
   • Reduced build context from ~2.1GB → ~45MB

✅ docker-compose.yml:
   • 3 services: fastapi_backend, streamlit_ui, redis_cache
   • Healt